# 01 - Exploração, Limpeza e Normalização de Dados

Este notebook realiza a limpeza, normalização e junção dos dados:
- **SINAN**: Dados epidemiológicos de doenças transmissíveis
- **INMET**: Dados meteorológicos
- **WBE**: Índices de Wastewater-Based Epidemiology

O resultado é um dataset consolidado salvo em `/data/processed` para uso no backend e IoT simulator.

## 1️⃣ Importar Bibliotecas Necessárias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')

# Configurar estilos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Definir caminhos
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

# Criar diretório processed se não existir
DATA_PROCESSED.mkdir(exist_ok=True)

print("✓ Bibliotecas importadas com sucesso!")
print(f"✓ Caminho de dados brutos: {DATA_RAW.absolute()}")
print(f"✓ Caminho de dados processados: {DATA_PROCESSED.absolute()}")

## 2️⃣ Carregar Dados Brutos do SINAN e INMET

Os dados devem estar em arquivos CSV dentro de `/data/raw`:
- `sinan_data.csv` - Dados epidemiológicos
- `inmet_data.csv` - Dados meteorológicos

In [ ]:
# Listar arquivos CSV disponíveis
csv_files = list(DATA_RAW.glob('*.csv'))
print(f"Arquivos CSV encontrados: {len(csv_files)}")
for file in csv_files:
    print(f"  - {file.name}")

# Tentar carregar dados se existirem
df_sinan = None
df_inmet = None

if (DATA_RAW / 'sinan_data.csv').exists():
    df_sinan = pd.read_csv(DATA_RAW / 'sinan_data.csv')
    print(f"\n✓ SINAN carregado: {df_sinan.shape}")
    print(df_sinan.head())
    print(f"\nTipos de dados SINAN:")
    print(df_sinan.dtypes)
else:
    print("\n✗ Arquivo sinan_data.csv não encontrado em /data/raw")

if (DATA_RAW / 'inmet_data.csv').exists():
    df_inmet = pd.read_csv(DATA_RAW / 'inmet_data.csv')
    print(f"\n✓ INMET carregado: {df_inmet.shape}")
    print(df_inmet.head())
    print(f"\nTipos de dados INMET:")
    print(df_inmet.dtypes)
else:
    print("\n✗ Arquivo inmet_data.csv não encontrado em /data/raw")

## 3️⃣ Limpeza de Dados e Tratamento de Valores Ausentes

In [ ]:
def analizar_valores_ausentes(df, nome_dataset):
    """Analisa e reporta valores ausentes em um dataframe"""
    print(f"\n{'='*60}")
    print(f"Análise de Valores Ausentes - {nome_dataset}")
    print(f"{'='*60}")
    
    missing = df.isnull().sum()
    missing_percent = (df.isnull().sum() / len(df)) * 100
    
    missing_df = pd.DataFrame({
        'Coluna': df.columns,
        'Valores_Ausentes': missing.values,
        'Percentual': missing_percent.values
    })
    
    missing_df = missing_df[missing_df['Valores_Ausentes'] > 0].sort_values('Percentual', ascending=False)
    
    if len(missing_df) == 0:
        print("✓ Nenhum valor ausente encontrado!")
    else:
        print(missing_df.to_string(index=False))
    
    return missing_df

# Aplicar análise nos datasets
if df_sinan is not None:
    analizar_valores_ausentes(df_sinan, 'SINAN')

if df_inmet is not None:
    analizar_valores_ausentes(df_inmet, 'INMET')

# Função para limpar dados
def limpar_dataset(df, nome):
    """Limpa um dataset removendo ou imputando valores ausentes"""
    print(f"\n{'='*60}")
    print(f"Limpeza de Dados - {nome}")
    print(f"{'='*60}")
    
    rows_antes = len(df)
    
    # Remover linhas com todos os valores ausentes
    df = df.dropna(how='all')
    
    # Preencher valores ausentes com a média (para colunas numéricas)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].mean(), inplace=True)
    
    # Preencher com forward fill (para séries temporais)
    df = df.fillna(method='ffill').fillna(method='bfill')
    
    rows_depois = len(df)
    print(f"Linhas removidas: {rows_antes - rows_depois}")
    print(f"✓ Dataset após limpeza: {df.shape}")
    
    return df

# Limpar datasets
if df_sinan is not None:
    df_sinan = limpar_dataset(df_sinan, 'SINAN')

if df_inmet is not None:
    df_inmet = limpar_dataset(df_inmet, 'INMET')

## 4️⃣ Normalização e Padronização de Dados

In [ ]:
def normalizar_dataset(df, nome):
    """Normaliza dados numéricos usando MinMaxScaler (0-1)"""
    print(f"\n{'='*60}")
    print(f"Normalização de Dados - {nome}")
    print(f"{'='*60}")
    
    df_norm = df.copy()
    numeric_cols = df_norm.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        scaler = MinMaxScaler()
        df_norm[numeric_cols] = scaler.fit_transform(df_norm[numeric_cols])
        print(f"✓ Colunas normalizadas: {list(numeric_cols)}")
        print(f"✓ Intervalo: [0, 1]")
    
    return df_norm

def padronizar_dataset(df, nome):
    """Padroniza dados numéricos usando StandardScaler (média=0, desvio=1)"""
    print(f"\nPadronização de Dados - {nome}")
    
    df_padr = df.copy()
    numeric_cols = df_padr.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        scaler = StandardScaler()
        df_padr[numeric_cols] = scaler.fit_transform(df_padr[numeric_cols])
        print(f"✓ Colunas padronizadas: {list(numeric_cols)}")
        print(f"✓ Distribuição: Média≈0, Desvio Padrão≈1")
    
    return df_padr

# Aplicar normalização
if df_sinan is not None:
    df_sinan_norm = normalizar_dataset(df_sinan, 'SINAN')

if df_inmet is not None:
    df_inmet_norm = normalizar_dataset(df_inmet, 'INMET')

## 5️⃣ Merge dos Datasets SINAN e INMET

Juntar os dados por **data** e **localização** para criar um dataset consolidado.

In [ ]:
# Merge de SINAN e INMET
df_merged = None

if df_sinan is not None and df_inmet is not None:
    print("Realizando merge dos datasets...")
    
    # Supor que ambos têm colunas 'data' e 'localizacao' (ajustar conforme necessário)
    # Exemplo: df_merged = pd.merge(df_sinan, df_inmet, on=['data', 'localizacao'], how='inner')
    
    # Para fins de demonstração, se os datasets não tiverem essas colunas:
    # Você pode usar merge por índice ou criar chaves de merge appropriadas
    
    try:
        # Tentar merge por data e localização (ajuste os nomes das colunas)
        key_columns = [col for col in df_sinan.columns if col in df_inmet.columns 
                       and col.lower() in ['data', 'date', 'localizacao', 'location']]
        
        if key_columns:
            df_merged = pd.merge(df_sinan, df_inmet, on=key_columns, how='inner')
            print(f"✓ Merge realizado com sucesso!")
            print(f"  Chaves de merge: {key_columns}")
        else:
            # Se não houver colunas comuns, fazer merge por índice
            print("⚠ Nenhuma coluna comum encontrada. Tentando merge por índice...")
            df_merged = pd.concat([df_sinan, df_inmet], axis=1)
            print("✓ Datasets concatenados por colunas")
        
        print(f"Dataset merged: {df_merged.shape}")
        print(f"\nPrimeiras linhas:")
        print(df_merged.head())
        
    except Exception as e:
        print(f"✗ Erro ao fazer merge: {e}")
        print("⚠ Usando concatenação por coluna como alternativa")
        df_merged = pd.concat([df_sinan, df_inmet], axis=1)
        
elif df_sinan is not None:
    print("Apenas SINAN disponível")
    df_merged = df_sinan.copy()
elif df_inmet is not None:
    print("Apenas INMET disponível")
    df_merged = df_inmet.copy()
else:
    print("✗ Nenhum dataset foi carregado!")

## 6️⃣ Análise Exploratória de Dados (EDA)

In [ ]:
if df_merged is not None:
    print(f"{'='*60}")
    print("ANÁLISE EXPLORATÓRIA DE DADOS")
    print(f"{'='*60}\n")
    
    # Estatísticas descritivas
    print("Estatísticas Descritivas:")
    print(df_merged.describe())
    
    # Visuzalizar dados numéricos
    numeric_cols = df_merged.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        # Histogramas
        print("\n\n📊 Distribuição das Variáveis Numéricas:")
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        
        for i, col in enumerate(numeric_cols[:4]):
            axes[i].hist(df_merged[col], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
            axes[i].set_title(f'Distribuição de {col}')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Frequência')
        
        plt.tight_layout()
        plt.show()
        
        # Box plots
        print("\n📦 Box plots para detecção de outliers:")
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        for i, col in enumerate(numeric_cols[:2]):
            axes[i].boxplot(df_merged[col])
            axes[i].set_title(f'Box Plot - {col}')
            axes[i].set_ylabel(col)
        
        plt.tight_layout()
        plt.show()
        
        # Matriz de correlação
        if len(numeric_cols) > 1:
            print("\n🔗 Matriz de Correlação:")
            correlation = df_merged[numeric_cols].corr()
            
            plt.figure(figsize=(10, 8))
            sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
                       cbar_kws={'label': 'Correlação'})
            plt.title('Correlação entre Variáveis')
            plt.tight_layout()
            plt.show()
else:
    print("✗ Nenhum dataset para análise")

## 7️⃣ Salvar Dados Processados

Exportar o dataset consolidado e limpo para `/data/processed` para uso no backend.

In [ ]:
if df_merged is not None:
    print(f"{'='*60}")
    print("SALVANDO DADOS PROCESSADOS")
    print(f"{'='*60}\n")
    
    # Salvar dataset consolidado
    output_file_merged = DATA_PROCESSED / 'merged_data.csv'
    df_merged.to_csv(output_file_merged, index=False)
    print(f"✓ Dados merged salvos em: {output_file_merged}")
    
    # Salvar dataset normalizado
    if df_sinan is not None:
        output_file_sinan = DATA_PROCESSED / 'sinan_processed.csv'
        df_sinan.to_csv(output_file_sinan, index=False)
        print(f"✓ SINAN processado salvo em: {output_file_sinan}")
    
    if df_inmet is not None:
        output_file_inmet = DATA_PROCESSED / 'inmet_processed.csv'
        df_inmet.to_csv(output_file_inmet, index=False)
        print(f"✓ INMET processado salvo em: {output_file_inmet}")
    
    # Gerar relatório de resumo
    summary_report = f"""
RELATÓRIO DE PROCESSAMENTO DE DADOS
====================================

Data de Processamento: {pd.Timestamp.now()}

DATASETS PROCESSADOS:
- SINAN: {df_sinan.shape if df_sinan is not None else 'N/A'}
- INMET: {df_inmet.shape if df_inmet is not None else 'N/A'}
- Merged: {df_merged.shape}

ARQUIVOS SALVOS:
- {output_file_merged}

PRÓXIMOS PASSOS:
1. Usar os dados em /data/processed no backend
2. Configurar o MQTT para receber dados do IoT Simulator
3. Implementar pipeline de armazenamento em banco de dados
"""
    
    print(summary_report)
    
else:
    print("✗ Nenhum dado para salvar")